# Session 8 — Tool Use and Function Calling

LLMs are excellent at reasoning but cannot act on the world directly. Tool use (also called function calling) gives the model a way to request external actions — fetching live data, running calculations, searching a document store — and incorporate the results into its answer.

This notebook covers:
1. Anatomy of a tool definition (JSON schema)
2. A custom weather tool — the classic first example
3. The agentic loop step by step
4. Built-in tools: `web_search` and `code_interpreter`
5. Parallel tool calls — two tools in one turn
6. RAG as a tool call — the model decides when to retrieve

## Learning Goals

- understand the JSON schema format for tool definitions
- implement the four-step agentic loop (request → detect → execute → feed back)
- use built-in OpenAI tools without writing a Python function
- define multiple tools and handle parallel calls
- refactor the RAG pipeline from notebook 04 so the model controls retrieval

In [ ]:
import json
import os
import requests
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_ORG_ID = os.getenv("OPENAI_ORG_ID")
OPENAI_PROJECT_ID = os.getenv("OPENAI_PROJECT_ID")

print("OpenAI key present:", bool(OPENAI_API_KEY))


def make_openai_client(api_key=None, base_url=None):
    kwargs = {}
    if api_key:
        kwargs["api_key"] = api_key
    if base_url:
        kwargs["base_url"] = base_url
    if api_key == OPENAI_API_KEY and not base_url:
        if OPENAI_ORG_ID:
            kwargs["organization"] = OPENAI_ORG_ID
        if OPENAI_PROJECT_ID:
            kwargs["project"] = OPENAI_PROJECT_ID
    return OpenAI(**kwargs)


def require_env(name: str):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


def session8_dir() -> Path:
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        return cwd.parent
    candidate = cwd / "material" / "Session 8"
    if candidate.exists():
        return candidate
    return cwd